# 05 — Evaluation & Results Visualization

Load evaluation result JSONs from `results/`, visualize Track A (retrieval)
and Track B (RAGAS) metrics, and compare across configurations.

Run new evaluations with:
```bash
python -m app.evaluate --mode full --chunk-config chunk800 --retrieval-only
python -m app.evaluate --mode embedding --chunk-config chunk400
```

In [ ]:
import sys, os, json, glob
from pathlib import Path

repo_root = str(Path(os.getcwd()).parents[1] / "RAG_Ai_Assistant_Uni")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

RESULTS_DIR = Path(repo_root) / "results"
print(f"Results dir: {RESULTS_DIR}")
result_files = sorted(RESULTS_DIR.glob("*.json"))
print(f"Found {len(result_files)} result files:")
for f in result_files:
    print(f"  {f.name}")

## 1. Load and Parse Results

In [ ]:
def parse_result(path: Path) -> dict | None:
    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    ret = data.get("retrieval", {})
    gen = data.get("generation", {})
    if not ret.get("metrics"):
        return None

    m = ret["metrics"]
    row = {
        "file":      path.name,
        "mode":      data.get("mode", "?"),
        "chunk":     data.get("chunk_config", "?"),
        "timestamp": data.get("timestamp", "")[:19],
        "n":         m.get("n_questions"),
        "hit@1":     m.get("hit_rate", {}).get("@1"),
        "hit@3":     m.get("hit_rate", {}).get("@3"),
        "hit@5":     m.get("hit_rate", {}).get("@5"),
        "mrr":       m.get("mrr"),
        "p@5":       m.get("precision", {}).get("@5"),
    }

    if gen and gen.get("metrics"):
        gm = gen["metrics"]
        row.update({
            "faithfulness":      gm.get("faithfulness"),
            "answer_relevancy":  gm.get("answer_relevancy"),
            "context_precision": gm.get("context_precision"),
            "context_recall":    gm.get("context_recall"),
        })

    return row

rows = [r for f in result_files if (r := parse_result(f)) is not None]
df = pd.DataFrame(rows)

if df.empty:
    print("No parseable result files found. Run app.evaluate first.")
else:
    display(df.drop(columns=["file", "timestamp"]).sort_values("mrr", ascending=False))

## 2. Track A — Retrieval Metrics

In [ ]:
if not df.empty:
    metrics_a = ["hit@1", "hit@3", "hit@5", "mrr", "p@5"]
    df_a = df.dropna(subset=["hit@1"]).copy()
    df_a["label"] = df_a["mode"] + " / " + df_a["chunk"]

    fig, axes = plt.subplots(1, len(metrics_a), figsize=(18, 4))
    for ax, metric in zip(axes, metrics_a):
        bars = ax.barh(df_a["label"], df_a[metric], edgecolor="black")
        ax.set_title(metric.upper())
        ax.set_xlim(0, 1.05)
        ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        for bar, val in zip(bars, df_a[metric]):
            if pd.notna(val):
                ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                        f"{val:.2f}", va="center", fontsize=8)
    plt.suptitle("Track A — Retrieval Metrics", fontsize=13)
    plt.tight_layout()
    plt.show()

## 3. Track B — RAGAS Metrics

Only shown if Track B results exist (requires full eval run without `--retrieval-only`).

In [ ]:
RAGAS_METRICS = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]

if not df.empty and "faithfulness" in df.columns:
    df_b = df.dropna(subset=["faithfulness"]).copy()
    df_b["label"] = df_b["mode"] + " / " + df_b["chunk"]

    if df_b.empty:
        print("No Track B results yet. Run without --retrieval-only to generate.")
    else:
        fig, axes = plt.subplots(1, len(RAGAS_METRICS), figsize=(18, 4))
        for ax, metric in zip(axes, RAGAS_METRICS):
            bars = ax.barh(df_b["label"], df_b[metric], edgecolor="black", color="steelblue")
            ax.set_title(metric.replace("_", " ").title())
            ax.set_xlim(0, 1.05)
            ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
            for bar, val in zip(bars, df_b[metric]):
                if pd.notna(val):
                    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                            f"{val:.2f}", va="center", fontsize=8)
        plt.suptitle("Track B — RAGAS Metrics", fontsize=13)
        plt.tight_layout()
        plt.show()
else:
    print("No Track B data in loaded results.")

## 4. MRR vs Hit@5 Scatter

Quick Pareto view — top-right is best.

In [ ]:
if not df.empty:
    df_a = df.dropna(subset=["mrr", "hit@5"]).copy()
    df_a["label"] = df_a["mode"] + "\n" + df_a["chunk"]

    fig, ax = plt.subplots(figsize=(7, 5))
    scatter = ax.scatter(df_a["mrr"], df_a["hit@5"], s=80, zorder=3)
    for _, row in df_a.iterrows():
        ax.annotate(row["label"], (row["mrr"], row["hit@5"]),
                    textcoords="offset points", xytext=(6, 0), fontsize=8)
    ax.set_xlabel("MRR")
    ax.set_ylabel("Hit@5")
    ax.set_title("MRR vs Hit@5 — all configs")
    ax.set_xlim(0, 1.05)
    ax.set_ylim(0, 1.05)
    ax.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

## 5. Run Evaluation Directly

Trigger an evaluation run from here without leaving the notebook.

In [ ]:
# Adjust params as needed, then uncomment to run.
#
# from app.evaluate import main as run_eval
# import sys
#
# sys.argv = [
#     "evaluate",
#     "--mode",         "full",
#     "--chunk-config", "chunk800",
#     "--retrieval-only",
#     "--verbose",
# ]
# run_eval()
print("Uncomment the block above to run an evaluation.")